# Online Retail II — Data Cleaning & Transformation

Source: https://archive.ics.uci.edu/dataset/502/online%2Bretail

This notebook performs data cleaning and transformation on the UCI Online Retail II dataset based on the findings and requirements identified during data profiling.

The cleaning process addresses duplicate records, overlapping source data, transaction classifications, inconsistent data types, special transaction activity, and other conditions that could affect revenue analysis.

Transformations are applied using documented and reproducible rules while preserving the original source data. Validation checks are performed throughout the process to confirm the impact of each transformation and prepare a reliable analytical dataset for downstream revenue analysis.

#### Project Setup

In [3]:
# Install packages
from google.colab import drive
import pandas as pd

# Connect to Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Save file path of online retail data
file_path = "/content/drive/MyDrive/Working/Resume 📃/Portfolio/Retail Revenue Intelligence/Data/online_retail_raw.xlsx"

# Load all sheets
sheets = pd.read_excel(file_path, sheet_name=None)

# Combine them into one DataFrame
df = pd.concat(sheets.values(), ignore_index=True)

# Add a column to keep track of which tab the data came from
df = pd.concat(
    [sheet.assign(Source_Sheet=name) for name, sheet in sheets.items()],
    ignore_index=True
)

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Source_Sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


#### Data Type Standardization

#### Standardize `StockCode`
Data profiling identified that `StockCode` contains a mixture of integer and string values. Because `StockCode` functions as a product identifier rather than a numerical measure, it is standardized as a string to ensure consistent handling during cleaning, classification, and analysis.

In [5]:
df["StockCode"] = df["StockCode"].astype(str)
df["StockCode"].map(type).value_counts()

,count
StockCode,
<class 'str'>,1067371


#### Remove Cross-Sheet Overlap

Data profiling identified overlapping transaction data between the two source worksheets from December 1–9, 2010. The same 22,202 unique transaction records were found in both source sheets.

To prevent these transactions from being counted twice, the redundant cross-sheet records will be removed while preserving one copy of each transaction.

In [6]:
# Record row count before cleaning
rows_before = len(df)

# Identify redundant overlap rows from the second source sheet
cross_sheet_overlap = (
    (df["Source_Sheet"] == "Year 2010-2011") &
    (df["InvoiceDate"] >= "2010-12-01") &
    (df["InvoiceDate"] < "2010-12-10")
)

# Remove redundant cross-sheet overlap
df_clean = df.loc[~cross_sheet_overlap].copy()

# Validate
rows_after = len(df_clean)

print(f"Rows before: {rows_before:,}")
print(f"Rows removed: {rows_before - rows_after:,}")
print(f"Rows after: {rows_after:,}")

Rows before: 1,067,371
Rows removed: 22,523
Rows after: 1,044,848


In [7]:
df_clean.groupby("Source_Sheet")["InvoiceDate"].agg(["min", "max"])

,min,max
Source_Sheet,,
Year 2009-2010,2009-12-01 07:45:00,2010-12-09 20:01:00
Year 2010-2011,2010-12-10 09:33:00,2011-12-09 12:50:00


#### Review Within-Source Exact Duplicates

After removing the confirmed cross-sheet overlap, 11,812 duplicate rows remain. Because the dataset does not contain a unique transaction-line identifier, these records cannot be confirmed as erroneous and are retained to avoid removing potentially valid transaction activity.

In [10]:
transaction_fields = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
    "Source_Sheet"
]

within_source_duplicates = df_clean[
    df_clean.duplicated(
        subset=transaction_fields,
        keep=False
    )
].copy()

print(f"Duplicate records involved: {len(within_source_duplicates):,}")

within_source_duplicates.sort_values(
    ["Invoice", "StockCode", "InvoiceDate"]
).head(5)

Duplicate records involved: 22,813


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Source_Sheet
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,Year 2009-2010
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,Year 2009-2010
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010


#### Classify Transaction Types

Data profiling identified several transaction patterns that should be distinguished before revenue analysis.

All records were classified using the transaction patterns identified during profiling. The resulting categories distinguish standard sales from cancellations/returns, inventory adjustments, zero-price transactions, and bad-debt adjustments, allowing each type to be handled appropriately during revenue analysis.

In [20]:
df_clean.loc[
    df_clean["Invoice"].astype(str).str.startswith("A"),
    "Transaction_Type"
] = "Bad Debt Adjustment"

df_clean["Transaction_Type"].value_counts(dropna=False)


inventory_adjustment = (
    ~df_clean["Invoice"].astype(str).str.startswith(("A", "C")) &
    (df_clean["Quantity"] < 0) &
    (df_clean["Price"] == 0) &
    (df_clean["Customer ID"].isna())
)

df_clean.loc[
    inventory_adjustment,
    "Transaction_Type"
] = "Inventory Adjustment"


cancellation_return = (
    df_clean["Invoice"].astype(str).str.startswith("C") &
    (df_clean["Quantity"] < 0)
)

df_clean.loc[
    cancellation_return,
    "Transaction_Type"
] = "Cancellation/Return"


standard_sale = (
    df_clean["Transaction_Type"].isna() &
    (df_clean["Quantity"] > 0) &
    (df_clean["Price"] > 0)
)

df_clean.loc[
    standard_sale,
    "Transaction_Type"
] = "Standard Sale"


zero_price_transaction = (
    df_clean["Transaction_Type"].isna() &
    (df_clean["Quantity"] > 0) &
    (df_clean["Price"] == 0)
)

df_clean.loc[
    zero_price_transaction,
    "Transaction_Type"
] = "Zero-Price Transaction"


df_clean["Transaction_Type"].value_counts(dropna=False)


,count
Transaction_Type,
Standard Sale,1019654
Cancellation/Return,19164
Inventory Adjustment,3393
Zero-Price Transaction,2631
Bad Debt Adjustment,6


#### Classify Merchandise and Special Activity

Certain `StockCode` values represent postage, discounts, manual entries, fees, commissions, adjustments, and other non-standard activity rather than typical merchandise. A separate classification is created so these records can be distinguished without changing their transaction classification.

The classification identified 5,414 special-activity records, including 4,209 standard-sale transactions and 1,178 cancellations/returns. `Activity_Type` is retained separately from `Transaction_Type` so non-merchandise activity can be included or excluded appropriately depending on the analysis.

In [21]:
special_stockcodes = [
    "POST",
    "DOT",
    "M",
    "m",
    "D",
    "S",
    "BANK CHARGES",
    "ADJUST",
    "AMAZONFEE",
    "CRUK",
    "B"
]

df_clean["Activity_Type"] = "Merchandise"

df_clean.loc[
    df_clean["StockCode"].isin(special_stockcodes),
    "Activity_Type"
] = "Special Activity"

df_clean["Activity_Type"].value_counts()

,count
Activity_Type,
Merchandise,1039434
Special Activity,5414


In [22]:
df_clean[
    df_clean["Activity_Type"] == "Special Activity"
]["StockCode"].value_counts()

,count
StockCode,
POST,2086
DOT,1425
M,1398
D,173
S,102
BANK CHARGES,100
ADJUST,67
AMAZONFEE,36
CRUK,16


In [23]:
pd.crosstab(
    df_clean["Activity_Type"],
    df_clean["Transaction_Type"]
)

Transaction_Type,Bad Debt Adjustment,Cancellation/Return,Inventory Adjustment,Standard Sale,Zero-Price Transaction
Activity_Type,,,,,
Merchandise,0,17986,3393,1015445,2610
Special Activity,6,1178,0,4209,21


#### Create Revenue Measures

Revenue measures are created to support downstream analysis. `Line_Revenue` captures the original transaction value calculated as `Quantity × Price`, while `Analytical_Revenue` applies the transaction classifications established during cleaning to determine which values should contribute to revenue KPIs. This preserves the original transaction value while providing a separate measure for business analysis.

In [24]:
df_clean["Line_Revenue"] = (
    df_clean["Quantity"] * df_clean["Price"]
)

df_clean["Line_Revenue"].describe()

,Line_Revenue
count,1.044848e+06
mean,1.809810e+01
std,2.940201e+02
min,-1.684696e+05
25%,3.750000e+00
50%,9.900000e+00
75%,1.770000e+01
max,1.684696e+05


In [25]:
df_clean.groupby("Transaction_Type")["Line_Revenue"].agg(
    ["count", "sum", "min", "max"]
)

,count,sum,min,max
Transaction_Type,,,,
Bad Debt Adjustment,6,-1.476141e+05,-53594.360,11062.06
Cancellation/Return,19164,-1.465677e+06,-168469.600,-0.12
Inventory Adjustment,3393,0.000000e+00,-0.000,-0.00
Standard Sale,1019654,2.052305e+07,0.001,168469.60
Zero-Price Transaction,2631,0.000000e+00,0.000,0.00


In [26]:
# Identify transactions eligible for revenue analysis
df_clean["Revenue_Eligible"] = df_clean["Transaction_Type"].isin(
    ["Standard Sale", "Cancellation/Return"]
)

# Calculate analytical revenue
df_clean["Analytical_Revenue"] = df_clean["Line_Revenue"].where(
    df_clean["Revenue_Eligible"],
    0
)

In [29]:
df_clean.groupby("Transaction_Type")["Analytical_Revenue"].agg(
    ["count", "sum", "min", "max"]
)

,count,sum,min,max
Transaction_Type,,,,
Bad Debt Adjustment,6,0.000000e+00,0.000,0.00
Cancellation/Return,19164,-1.465677e+06,-168469.600,-0.12
Inventory Adjustment,3393,0.000000e+00,0.000,0.00
Standard Sale,1019654,2.052305e+07,0.001,168469.60
Zero-Price Transaction,2631,0.000000e+00,0.000,0.00


In [31]:
print(f"Total Line Revenue: ${df_clean['Line_Revenue'].sum():,.2f}")
print(f"Total Analytical Revenue: ${df_clean['Analytical_Revenue'].sum():,.2f}")

Total Line Revenue: $18,909,762.12
Total Analytical Revenue: $19,057,376.20


#### Validate Cleaned Dataset

Final validation is performed to confirm the structure and integrity of the cleaned dataset before export. Row counts, data types, missing values, transaction classifications, activity classifications, and date coverage are reviewed to ensure the applied transformations produced the expected results.

Final validation confirmed that the cleaned dataset contains 1,044,848 records with complete transaction and activity classifications and preserved date coverage from December 2009 through December 2011. Remaining missing values in `Customer ID` and `Description` were intentionally retained according to the documented cleaning rules. The dataset is now ready for downstream analysis.

In [ ]:
print(f"Final rows: {df_clean.shape[0]:,}")
print(f"Final columns: {df_clean.shape[1]}")

df_clean.info()

In [ ]:
df_clean.isna().sum()

In [34]:
print("Transaction Types:")
print(df_clean["Transaction_Type"].value_counts())

print("\nActivity Types:")
print(df_clean["Activity_Type"].value_counts())

print("\nDate Coverage:")
print(f"Earliest: {df_clean['InvoiceDate'].min()}")
print(f"Latest:   {df_clean['InvoiceDate'].max()}")

Transaction Types:
Transaction_Type
Standard Sale             1019654
Cancellation/Return         19164
Inventory Adjustment         3393
Zero-Price Transaction       2631
Bad Debt Adjustment             6
Name: count, dtype: int64

Activity Types:
Activity_Type
Merchandise         1039434
Special Activity       5414
Name: count, dtype: int64

Date Coverage:
Earliest: 2009-12-01 07:45:00
Latest:   2011-12-09 12:50:00


In [35]:
print(
    "Unclassified transactions:",
    df_clean["Transaction_Type"].isna().sum()
)

print(
    "Unclassified activity:",
    df_clean["Activity_Type"].isna().sum()
)

Unclassified transactions: 0
Unclassified activity: 0


#### Export Cleaned Dataset

In [36]:
output_path = "/content/drive/MyDrive/Working/Resume 📃/Portfolio/Retail Revenue Intelligence/Data/online_retail_clean.csv"

df_clean.to_csv(
    output_path,
    index=False
)

print(f"Clean dataset exported to:\n{output_path}")

Clean dataset exported to:
/content/drive/MyDrive/Working/Resume 📃/Portfolio/Retail Revenue Intelligence/Data/online_retail_clean.csv
